# I’m Something of a Painter Myself
Use GANs to create art - will you be the next Monet?

### Dataset Setup
This cell defines the training dataset class and prepares file paths for Monet and photo images.

In [ ]:
# Dataset
import os
if os.path.exists("/kaggle/input/competitions/gan-getting-started"):
    ROOT = "/kaggle/input/competitions/gan-getting-started"
else:
    ROOT = os.getcwd()

from PIL import Image
from torch.utils.data import Dataset, DataLoader

class MonetPhotoDataset(Dataset):
    def __init__(self, monet_dir: str, photo_dir: str, transform=None):
        super().__init__()
        self.monet_dir = monet_dir
        self.photo_dir = photo_dir
        self.transform = transform

        self.monet_images = sorted(os.listdir(monet_dir))
        self.photo_images = sorted(os.listdir(photo_dir))

        self.monet_len = len(self.monet_images)
        self.photo_len = len(self.photo_images)

        self.length_dataset = max(self.monet_len, self.photo_len)

    def __len__(self):
        return self.length_dataset
    
    def __getitem__(self, idx : int) -> tuple:
        monet_img = self.monet_images[idx % self.monet_len]
        photo_img = self.photo_images[idx % self.photo_len]

        monet_path = os.path.join(self.monet_dir, monet_img)
        photo_path = os.path.join(self.photo_dir, photo_img)

        monet_image = Image.open(monet_path).convert("RGB")
        photo_image = Image.open(photo_path).convert("RGB")

        if self.transform:
            monet_image = self.transform(monet_image)
            photo_image = self.transform(photo_image)

        return monet_image, photo_image

### Data Preprocessing
This cell configures image transforms used for training and testing, including normalization.

In [ ]:
# Preprocessing
import torchvision.transforms as transforms

# Train transform
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Aditional transforms for data augmentation
# train_transform = transforms.Compose([
#     transforms.Resize(286),
#     transforms.RandomCrop(256),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomRotation(10),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
# ])

# Test transform
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

### Dataset Verification
This cell performs a quick sanity check by loading a batch and visualizing sample images.

In [ ]:
# Verify the dataset
from matplotlib import pyplot as plt

monet_dir = os.path.join(ROOT, "monet_jpg")
photo_dir = os.path.join(ROOT, "photo_jpg")
dataset = MonetPhotoDataset(monet_dir, photo_dir, transform=train_transform)
dataloader = DataLoader(dataset, batch_size=6, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2, drop_last=True)

for monet_images, photo_images in dataloader:
    print(monet_images.shape)  # Should be [6, 3, 256, 256]
    print(photo_images.shape)  # Should be [6, 3, 256, 256]
    break

X, y = dataset[0]
print(X.shape, y.shape)  # Should be [3, 256, 256] [3, 256, 256]
plt.subplot(1, 2, 1)
plt.imshow((X.permute(1, 2, 0) * 0.5 + 0.5).numpy())  # Denormalize for visualization
plt.title("Monet Image")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow((y.permute(1, 2, 0) * 0.5 + 0.5).numpy())  # Denormalize for visualization
plt.title("Photo Image")
plt.axis("off")
plt.show()

### Generator Architecture
This cell defines core neural network components, including the residual block and generator model.

In [ ]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels : int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=0),
            nn.InstanceNorm2d(in_channels),
            nn.ReLU(inplace=True),

            nn.ReflectionPad2d(1),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=0),
            nn.InstanceNorm2d(in_channels)
        )
        return None

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        return x + self.block(x)
    
class Generator(nn.Module):
    def __init__(self, input_shape: tuple, num_residual_blocks: int = 9, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        channels, height, width = input_shape

        # Initial Convolution and Downsampling
        out_features = 64
        self.initial = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(channels, out_features, kernel_size=7, stride=1, padding=0),
            nn.InstanceNorm2d(out_features),
            nn.ReLU(inplace=True)
        )

        in_features = out_features
        for _ in range(2):
            out_features *= 2
            self.initial.add_module(
                f"downsample_{out_features}",
                nn.Sequential(
                    nn.Conv2d(in_features, out_features, kernel_size=3, stride=2, padding=1),
                    nn.InstanceNorm2d(out_features),
                    nn.ReLU(inplace=True)
                )
            )
            in_features = out_features
        
        # Residual Blocks
        self.residual_blocks = nn.Sequential(
            *[ResidualBlock(out_features) for _ in range(num_residual_blocks)]
        )

        # Upsampling
        for _ in range(2):
            out_features //= 2
            self.residual_blocks.add_module(
                f"upsample_{out_features}",
                nn.Sequential(
                    nn.ConvTranspose2d(in_features, out_features, kernel_size=4, stride=2, padding=1),
                    nn.InstanceNorm2d(out_features),
                    nn.ReLU(inplace=True)
                )
            )
            in_features = out_features
        
        # Final Convolution
        self.final = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(out_features, channels, kernel_size=7, stride=1, padding=0),
            nn.Tanh()
        )

        self.model = nn.Sequential(
            self.initial,
            self.residual_blocks,
            self.final
        )

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        return self.model(x)

### Generator Forward Test
This cell runs a quick forward pass to validate generator output tensor shape.

In [ ]:
# Test the generator
G = Generator(input_shape=(3, 256, 256))
test_result = G(torch.randn(6, 3, 256, 256))  # Add batch dimension
print("Input shape:", (6, 3, 256, 256))
print("Output shape:", test_result.shape)  # Should be [6, 3, 256, 256]

### Discriminator Architecture
This cell defines the PatchGAN-style discriminator used for adversarial training.

In [ ]:
# Discriminator
class Discriminator(nn.Module):
    def __init__(self, input_shape: tuple, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        channels, height, width = input_shape

        def discriminator_block(in_filters : int, out_filters : int, kernel_size : int = 4, stride : int = 2, padding : int = 1, normalize : bool = True) -> nn.Sequential:
            block = nn.Sequential(
                nn.Conv2d(in_filters, out_filters, kernel_size, stride, padding),
                nn.InstanceNorm2d(out_filters) if normalize else nn.Identity(),
                nn.LeakyReLU(0.2, inplace=True)
            )
            return block
            
        self.model = nn.Sequential(
            discriminator_block(channels, 64, stride=2, normalize=False),
            discriminator_block(64, 128, stride=2),
            discriminator_block(128, 256, stride=2),
            discriminator_block(256, 512, stride=2),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(512, 1, kernel_size=4, padding=1)
        )

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        return self.model(x)

### Discriminator Forward Test
This cell verifies the discriminator output shape on a random input batch.

In [ ]:
# Test the discriminator
D = Discriminator(input_shape=(3, 256, 256))
test_result = D(torch.randn(6, 3, 256, 256))  # Add batch dimension
print("Input shape:", (6, 3, 256, 256))
print("Output shape:", test_result.shape)  # Should be [6, 1, 16, 16]

### Training Utilities
This cell defines loss functions, mixed-precision training helpers, and visualization routines.

In [ ]:
# Loss functions
import itertools
from torch.optim import Adam
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import torchvision.utils as vutils

criterion_GAN = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_identity = nn.L1Loss()

# loss weight hyperparameters
lambda_cycle = 10.0
lambda_identity = 5.0
# Formula: L_G = L_GAN + lambda_cycle * L_cycle + lambda_identity * L_identity

scaler_G = GradScaler()
scaler_D_M = GradScaler()
scaler_D_P = GradScaler()

def train_one_epoch(
    dataloader : DataLoader,
    generator_M2P : Generator,
    generator_P2M : Generator,
    discriminator_M : Discriminator,
    discriminator_P : Discriminator,
    optimizer_G : Adam,
    optimizer_D_M : Adam,
    optimizer_D_P : Adam,
    scaler_G : GradScaler,
    scaler_D_M : GradScaler,
    scaler_D_P : GradScaler,
    criterion_GAN : nn.Module,
    criterion_cycle : nn.Module,
    criterion_identity : nn.Module,
    device : torch.device,
    lambda_cycle : float,
    lambda_identity : float,
) -> tuple[float, float]:
    generator_M2P.train()
    generator_P2M.train()
    discriminator_M.train()
    discriminator_P.train()

    total_loss_G = 0.0
    total_loss_D = 0.0

    pbar = tqdm(dataloader, desc="Training Epoch")

    for i, batch in enumerate(pbar):
        real_monet, real_photo = batch
        real_monet = real_monet.to(device)
        real_photo = real_photo.to(device)

        pred_real_M = discriminator_M(real_monet)
        valid_label = torch.ones_like(pred_real_M, device=device)
        fake_label = torch.zeros_like(pred_real_M, device=device)

        optimizer_G.zero_grad(set_to_none=True)
        with autocast('cuda'):
            identity_monet = generator_P2M(real_monet)
            loss_identity_monet = criterion_identity(identity_monet, real_monet) * lambda_identity

            identity_photo = generator_M2P(real_photo)
            loss_identity_photo = criterion_identity(identity_photo, real_photo) * lambda_identity

            fake_monet = generator_P2M(real_photo)
            loss_GAN_P2M = criterion_GAN(discriminator_M(fake_monet), valid_label)

            fake_photo = generator_M2P(real_monet)
            loss_GAN_M2P = criterion_GAN(discriminator_P(fake_photo), valid_label)

            recov_monet = generator_P2M(fake_photo)
            loss_cycle_monet = criterion_cycle(recov_monet, real_monet) * lambda_cycle

            recov_photo = generator_M2P(fake_monet)
            loss_cycle_photo = criterion_cycle(recov_photo, real_photo) * lambda_cycle

            loss_G = loss_identity_monet + loss_identity_photo + loss_GAN_P2M + loss_GAN_M2P + loss_cycle_monet + loss_cycle_photo

        scaler_G.scale(loss_G).backward()
        scaler_G.step(optimizer_G)
        scaler_G.update()

        optimizer_D_M.zero_grad(set_to_none=True)
        with autocast('cuda'):
            loss_real_M = criterion_GAN(discriminator_M(real_monet), valid_label)
            loss_fake_M = criterion_GAN(discriminator_M(fake_monet.detach()), fake_label)
            loss_D_M = (loss_real_M + loss_fake_M) * 0.5
        scaler_D_M.scale(loss_D_M).backward()
        scaler_D_M.step(optimizer_D_M)
        scaler_D_M.update()

        optimizer_D_P.zero_grad(set_to_none=True)
        with autocast('cuda'):
            loss_real_P = criterion_GAN(discriminator_P(real_photo), valid_label)
            loss_fake_P = criterion_GAN(discriminator_P(fake_photo.detach()), fake_label)
            loss_D_P = (loss_real_P + loss_fake_P) * 0.5
        scaler_D_P.scale(loss_D_P).backward()
        scaler_D_P.step(optimizer_D_P)
        scaler_D_P.update()

        total_loss_G += loss_G.item()
        total_loss_D += loss_D_M.item() + loss_D_P.item()

        pbar.set_postfix({
            "loss_G": f"{loss_G.item():.4f}",
            "loss_D": f"{(loss_D_M.item() + loss_D_P.item()):.4f}"
        })

    return total_loss_G / len(dataloader), total_loss_D / len(dataloader)

def evaluate_visually(
    generator_M2P : Generator,
    generator_P2M : Generator,
    dataloader : DataLoader,
    device : torch.device,
    num_images : int = 4
) -> None:
    generator_M2P.eval()
    generator_P2M.eval()

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            real_monet, real_photo = batch
            real_monet = real_monet.to(device)
            real_photo = real_photo.to(device)

            fake_monet = generator_P2M(real_photo)
            fake_photo = generator_M2P(real_monet)

            img_grid = vutils.make_grid(
                torch.cat((real_monet[:num_images], fake_photo[:num_images], real_photo[:num_images], fake_monet[:num_images]), dim=0),
                nrow=num_images,
                normalize=True,
                scale_each=False,
                value_range=(-1, 1)
            )

            plt.figure(figsize=(12, 6))
            plt.imshow(img_grid.permute(1, 2, 0).cpu().numpy())
            plt.title(
                "Row 1: Real Monet  |  Row 2: Fake Photo (Generated)\n"
                "Row 3: Real Photo  |  Row 4: Fake Monet (Generated)",
                fontsize=14, pad=20
            )
            plt.axis("off")
            plt.show()
            break

    generator_M2P.train()
    generator_P2M.train()
    return None

### Model Training Loop
This cell initializes models/optimizers and runs the epoch-based CycleGAN training process.

In [ ]:
# Train the model
import torch.backends.cudnn as cudnn
cudnn.benchmark = True  # Enable cuDNN auto-tuner for better performance

# Initialize models, optimizers, and loss functions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count()
print(f"Current computing device in use: {device}")
if device.type == 'cuda':
    print(f"Number of available GPUs detected: {gpu_count}")

generator_M2P = Generator(input_shape=(3, 256, 256)).to(device)
generator_P2M = Generator(input_shape=(3, 256, 256)).to(device)
discriminator_M = Discriminator(input_shape=(3, 256, 256)).to(device)
discriminator_P = Discriminator(input_shape=(3, 256, 256)).to(device)

optimizer_G = Adam(itertools.chain(generator_M2P.parameters(), generator_P2M.parameters()), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_M = Adam(discriminator_M.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_P = Adam(discriminator_P.parameters(), lr=0.0002, betas=(0.5, 0.999))

def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)

num_epochs = 0
for epoch in range(num_epochs):
    avg_loss_G, avg_loss_D = train_one_epoch(
        dataloader,
        generator_M2P,
        generator_P2M,
        discriminator_M,
        discriminator_P,
        optimizer_G,
        optimizer_D_M,
        optimizer_D_P,
        scaler_G,
        scaler_D_M,
        scaler_D_P,
        criterion_GAN,
        criterion_cycle,
        criterion_identity,
        device,
        lambda_cycle,
        lambda_identity
    )
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss G: {avg_loss_G:.4f}, Average Loss D: {avg_loss_D:.4f}")

    # Evaluate visually every 5 epochs
    if (epoch + 1) % 5 == 0:
        evaluate_visually(generator_M2P, generator_P2M, dataloader, device)

### Test Data Pipeline
This cell prepares the test dataset class and dataloader for inference.

In [ ]:
# Test Dataset and Dataloader
class TestMonetPhotoDataset(Dataset):
    def __init__(self, photo_dir: str, transform=None) -> None:
        super().__init__()
        self.photo_dir = photo_dir
        self.transform = transform

        self.photo_images = sorted(os.listdir(photo_dir))

    def __len__(self) -> int:
        return len(self.photo_images)
    
    def __getitem__(self, idx : int) -> torch.Tensor:
        photo_name = self.photo_images[idx]
        photo_path = os.path.join(self.photo_dir, photo_name)
        photo_image = Image.open(photo_path).convert("RGB")

        if self.transform:
            photo_image = self.transform(photo_image)
        else:
            photo_image = transforms.ToTensor()(photo_image)

        return photo_image

### Inference and Export
This cell translates photos to Monet-style outputs and packs generated images into a zip file.

In [ ]:
# Inference function
def translate_image(
    generator_P2M : Generator,
    photo_dataloader : DataLoader,
    device : torch.device,
    output_dir : str = "./generated_images",
    max_images : int = 7038
) -> None:
    os.makedirs(output_dir, exist_ok=True)

    generator_P2M.eval()
    img_count = 0

    with torch.no_grad():
        for batch in tqdm(photo_dataloader, desc="Translating to Monet Style"):
            real_photos = batch
            real_photos = real_photos.to(device)

            fake_monets = generator_P2M(real_photos)

            fake_monets = (fake_monets * 0.5) + 0.5
            fake_monets = (fake_monets * 255).clamp(0, 255).to(torch.uint8)
            fake_monets = fake_monets.cpu().numpy()

            for i in range(fake_monets.shape[0]):
                img_array = fake_monets[i].transpose(1, 2, 0)
                img = Image.fromarray(img_array)
                img.save(os.path.join(output_dir, f"generated_{img_count:04d}.jpg"), format="JPEG")

                img_count += 1
                if img_count >= max_images:
                    break
            if img_count >= max_images:
                break

# create zip file of generated images
import zipfile

def create_zip_from_directory(directory : str, zip_name : str) -> None:
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        for foldername, subfolders, filenames in os.walk(directory):
            for filename in filenames:
                file_path = os.path.join(foldername, filename)
                zipf.write(file_path, os.path.relpath(file_path, directory))

    return None

### Final Generation Step
This final cell runs inference end-to-end and writes the submission archive file.

In [ ]:
# We do not run this cell directly as the notebook environment does not support Multi-GPU parallel training, instead, we run a python file.
# generate_img_dir = "./generated_images"
# test_dataset = TestMonetPhotoDataset(photo_dir=photo_dir, transform=test_transform)
# test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)
# translate_image(generator_P2M, test_dataloader, device=device, output_dir=generate_img_dir, max_images=7038)
# create_zip_from_directory(generate_img_dir, "images.zip")

import torch
import gc

# Free all GPU memory from notebook before launching torchrun
# Delete GPU models and optimizers
del generator_M2P, generator_P2M, discriminator_M, discriminator_P
del optimizer_G, optimizer_D_M, optimizer_D_P
del scaler_G, scaler_D_M, scaler_D_P
# Delete CPU test variables from earlier cells
del G, D, test_result
# Delete dataloader (terminates persistent workers) and dataset
del dataloader, dataset
gc.collect()
torch.cuda.empty_cache()

!torchrun --nproc_per_node=2 gan-getting-started.py